In [ ]:
!pip install -q selenium webdriver-manager pandas

In [ ]:

import pandas as pd 
from selenium.common.exceptions import TimeoutException, NoSuchElementException 
from datetime import date, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
import time, random, csv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time, re, os
from datetime import datetime
from calendar import monthrange

In [ ]:
def guardar_a_csv(datos, nombre_archivo="vuelos_kayak_data.csv"):
    """
    Convert a list of dictionaries to panda datframe and stores it in a csv file
    """
    if not datos:
        print("empty list. cannot store in csv")
        return

   
    df = pd.DataFrame(datos)
    
    columnas_ordenadas = [
        'fecha_vuelo',
        'precio', 
        'aerolinea', 
        'hora_salida', 
        'hora_llegada', 
        'origen_iata', 
        'destino_iata', 
        'paradas'
    ]
    

    columnas_a_usar = [col for col in columnas_ordenadas if col in df.columns]
    df = df[columnas_a_usar]

    df.to_csv(
        nombre_archivo,
        mode='a',            
        header=not os.path.exists(nombre_archivo),  
        index=False,
        encoding='utf-8',
        sep=';'
)
    
    

In [ ]:
cities = ["MAD","LON","PAR","BER","ROM","BUH","VIE","AMS","CPH","STO","OSL","HEL","LIS","IST","ZRH","BRU","DUB","PRG","BUD","ATH", "MIL"]

#pairs = [(o,d) for o in cities for d in cities if o != d]
pairs = [(cities[i], cities[j]) for i in range(len(cities)) for j in range(i+1, len(cities))]
# 1 departure date each 6 days
start = date(2025,12,1)
end = date(2026,8,31)
delta = (end - start).days
depart_dates = [start + timedelta(days=i) for i in range(0, delta+1, 6)] #Then we did not use these and use 3 days a month instead:

random.seed(42)  
start_year, start_month = 2025, 12
end_year, end_month = 2026, 8
k_per_month = 3   # <-- 3 random dates each month

# generates month list between december 2025 and august 2026
months = []
y, m = start_year, start_month
while (y < end_year) or (y == end_year and m <= end_month):
    months.append((y, m))
    if m == 12:
        y += 1
        m = 1
    else:
        m += 1

# -----------------------------------
# Dates per month
# -----------------------------------

def fechas_por_mes(year, month, k):
    days_in_month = monthrange(year, month)[1]
    all_dates = [date(year, month, d) for d in range(1, days_in_month + 1)]
    if len(all_dates) <= k:
        return all_dates
    return sorted(random.sample(all_dates, k))

# ---------------------------------------------
#  3 dates per month for each city pair
# ---------------------------------------------

fechas_por_par = {}

for origin, dest in pairs:
    lista_fechas = []
    for (y,m) in months:
        fechas_m = fechas_por_mes(y, m, k_per_month)
        lista_fechas.extend(fechas_m)
    fechas_por_par[(origin, dest)] = sorted(lista_fechas)

print(fechas_por_par[(pairs[2][0], pairs[2][1])][4])

def build_kayak_url(o, d, dep):
    dep_s = dep.strftime("%Y-%m-%d")
    return f"https://www.kayak.es/flights/{o}-{d}/{dep_s}?sort=bestflight_a"

# This is just to show how the url building works
url = build_kayak_url(pairs[2][0], pairs[2][1], depart_dates[6])
print(url)

In [ ]:


# 1) Chrome options. No local header to avoid getting detected as a bot
opts = Options()  

# Anti-detección útiles
opts.add_argument("--disable-blink-features=AutomationControlled")
opts.add_experimental_option("excludeSwitches", ["enable-automation"])
opts.add_experimental_option("useAutomationExtension", False)
opts.add_argument("--window-size=1366,768")
opts.add_argument("--start-maximized")
opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36")

# 2) Initiate driver with webdriver-manager
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=opts)

driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": """
        Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
        window.navigator.chrome = { runtime: {} };
        Object.defineProperty(navigator, 'languages', {get: () => ['en-US','en']});
        Object.defineProperty(navigator, 'plugins', {get: () => [1,2,3]});
    """
})

def click_Aceptar_todo(timeout=40):
    # 1) Try to find it by XPATH 
    xpath_text = "//*[normalize-space(text())='Aceptar todo']"
    try:
        el = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.XPATH, xpath_text))
        )
        clase_del_elemento = el.get_attribute('class')
        print(f" Encontrado: Elemento con texto 'Aceptar todo'.")
        print(f"   Clase CSS del elemento: **{clase_del_elemento}**")
        # If div eis inside <button>, click ancestor <button>
        try:
            btn = el.find_element(By.XPATH, "ancestor::button[1]")
            btn.click()
        except Exception:
            # If there is no button, click div
            el.click()
        print("Clicked 'Aceptar todo' (direct match).")
        return True
    except TimeoutException:
        pass

    # 2) Try to search it by class (fallback)
    try:
        el2 = WebDriverWait(driver, 2).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.RxNS-button-content"))
        )
        # Make sure it is the text we are looking for
        if el2.text.strip().lower() == "aceptar todo":
            try:
                btn = el2.find_element(By.XPATH, "ancestor::button[1]")
                btn.click()
            except Exception:
                el2.click()
            print("Clicked 'Aceptar todo' (by class).")
            return True
    except TimeoutException:
        pass

    # 3) If it does not show in main DOM, search iframes
    frames = driver.find_elements(By.TAG_NAME, "iframe")
    for f in frames:
        try:
            driver.switch_to.frame(f)
            try:
                elf = driver.find_element(By.XPATH, xpath_text)
                try:
                    btn = elf.find_element(By.XPATH, "ancestor::button[1]")
                    btn.click()
                except Exception:
                    elf.click()
                driver.switch_to.default_content()
                print("Clicked 'Aceptar todo' inside iframe.")
                return True
            except Exception:
                driver.switch_to.default_content()
                continue
        except Exception:
            driver.switch_to.default_content()
            continue

    print("No se encontró el botón 'Aceptar todo'.")
    return False


def esperar_carga_vuelos(driver, contenedor_clase, timeout=20):
    """
    Wait until there is accesible flights info
    """
    print("Esperando la carga completa de vuelos...")
    
    # 1. Wait until first spinner dissapears
    try:
        WebDriverWait(driver, timeout).until_not(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'div.NnB_N'))
        )
        print("Spinner de carga desaparecido.")
    except TimeoutException:
        print("Timeout esperando que desaparezca el spinner.")

    # 2. Wait until first flight info appears
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, f"div.{contenedor_clase}"))
        )
        return True
    except TimeoutException:
        print(f"Error: No se encontraron resultados de vuelos con la clase '{contenedor_clase}' después de la espera.")
        return False


def extraer_contenedores(driver, contenedor_clase):
    """
    Finds all flights in the containers
    """
    try:
        # Find flights by the container class
        contenedores = driver.find_elements(By.CSS_SELECTOR, f"div.{contenedor_clase}")
        return contenedores
        
    except Exception as e:
        print(f"Error al buscar contenedores: {e}")
        return []
        

def extraer_datos_vuelo_completo(contenedor, fecha_vuelo_global): 
    """
    Extracts flights info by the container, using price as a filter to discard ads and assigning the global date
    """
    # Classes for the extraction
    CLASE_PRECIO = "e2GB-price-text"
    CLASE_HORARIOS_CONT = "vmXl.vmXl-mod-variant-large"
    CLASE_AEROLINEA = "J0g6-operator-text"
    CLASE_CIUDAD_SALIDA = "jLhY-airport-info"
    CLASE_DIRECTO_CONT = "vmXl.vmXl-mod-variant-default"

    # Dictionary for saving the data
    vuelo = {
        'tipo': 'VUELO',
        'fecha_vuelo': fecha_vuelo_global,
        'precio': None,
        'hora_salida': 'N/A',
        'hora_llegada': 'N/A',
        'aerolinea': 'N/A',
        'origen_iata': 'N/A',
        'destino_iata': 'N/A',
        'paradas': 'N/A'
    }

    try:
        # ----------------------------------------------------
        # 1. Filtering ads by price
        # ----------------------------------------------------
        precio_el = contenedor.find_element(By.CSS_SELECTOR, f"div.{CLASE_PRECIO}")
        
        # --- PRECIO (Limpieza) ---
        precio_bruto = precio_el.text.strip()
        precio_limpio = re.sub(r'[^\d,]', '', precio_bruto).replace(',', '.')
        try:
            vuelo['precio'] = float(precio_limpio)
        except ValueError:
            vuelo['precio'] = precio_bruto 
            
        # ----------------------------------------------------
        # 2. Get the rest of the variables
        # ----------------------------------------------------

        # --- Departure and Landing times ---
        try:
            horarios_cont = contenedor.find_element(By.CSS_SELECTOR, f"div.{CLASE_HORARIOS_CONT.replace(' ', '.')}")
            tiempos_span = horarios_cont.find_elements(By.TAG_NAME, "span")
            if len(tiempos_span) >= 1:
                vuelo['hora_salida'] = tiempos_span[0].text.strip()
            if len(tiempos_span) >= 3:
                vuelo['hora_llegada'] = tiempos_span[2].text.strip()
        except NoSuchElementException:
            pass # if it fales it uses N/A

        # --- Airline ---
        try: 
            aerolinea_el = contenedor.find_element(By.CSS_SELECTOR, f"div.{CLASE_AEROLINEA}")
            vuelo['aerolinea'] = aerolinea_el.text.strip()
        except NoSuchElementException:
            pass

        # --- Origin and Dest (IATA code) ---
        try: 
            # Search for both at the same time
            destino_info_spans = contenedor.find_elements(By.CSS_SELECTOR, f"span.{CLASE_CIUDAD_SALIDA}")
            
            # Origin (first span)
            if len(destino_info_spans) >= 1:
                origen_iata_el = destino_info_spans[0].find_element(By.TAG_NAME, "span")
                vuelo['origen_iata'] = origen_iata_el.text.strip()

            # Dest (second span)
            if len(destino_info_spans) >= 2:
                 destino_iata_el = destino_info_spans[1].find_element(By.TAG_NAME, "span")
                 vuelo['destino_iata'] = destino_iata_el.text.strip()
        except NoSuchElementException:
            pass
            
        # --- Stops (flight scales) ---
        try: 
            escalas_el = contenedor.find_element(By.CSS_SELECTOR, f"div.{CLASE_DIRECTO_CONT.replace(' ', '.')}")
            vuelo['paradas'] = escalas_el.text.strip()
        except NoSuchElementException:
            pass
            
        return vuelo
        
    except NoSuchElementException:
        #This only applies when it does not pass the ad filter by price
        return {'tipo': 'ANUNCIO/PROMO'}
    
    except Exception as e:
        print(f"Error inesperado al procesar un contenedor: {e}")
        return {'tipo': 'ERROR'}


CLASE_CONTENEDOR_FECHA = "NITa-date-inline"
CLASE_VALOR_FECHA = "NITa-value"
def extraer_fecha_global(driver, clase_contenedor, clase_valor, timeout=5):
    """
    Search for the flight date:
    1. Search in main container (div.NITa-date-inline).
    2. Search for the span span.NITa-value inside that container
    """
    try:
        # 1. Wait until it appears
        contenedor_el = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, f"div.{clase_contenedor}"))
        )
        
        
        # 2. Search for the value
        fecha_el = contenedor_el.find_element(By.CSS_SELECTOR, f"span.{clase_valor}")
        
        fecha_texto = fecha_el.text.strip()
        return fecha_texto
        
    except TimeoutException:
        print(f"El contenedor de fecha 'div.{clase_contenedor}' no apareció después de {timeout} segundos.")
        return 'N/A'
    except NoSuchElementException:
        # Esto ocurre si el contenedor se encontró, pero el span.NITa-value no estaba dentro (improbable en este caso, pero seguro).
        print(f"Error: El elemento de valor 'span.{clase_valor}' no se encontró dentro del contenedor.")
        return 'N/A'

#This variables are made to start from a point in the list (not from the beggining)
start_origin = "DUB"
start_dest = "MIL"  
start_date = date(2026, 5, 15)
skip = True
i = 0
for origin, dest in pairs:                     # 1. Loop by cities
    for dep in fechas_por_par[(origin, dest)]:                  # 2. Loop by dates

        # Skip until the desired pair and date
        if skip:
            if origin == start_origin and dest == start_dest and dep >= start_date:
                skip = False   
                print("\nReanudando aquí:", origin, dest, dep)
            else:
                continue

        
        url = build_kayak_url(origin, dest, dep)   # get the url
        print("Scrapeando:", url)

        try:
            driver.get(url)
            if (i == 0):
                click_Aceptar_todo()    # we just search for the cookies banner once, this will save us from the timeout
                i = 1
            time.sleep(1.5)

            # extract global date only once    
            fecha_de_busqueda = extraer_fecha_global(driver, CLASE_CONTENEDOR_FECHA, CLASE_VALOR_FECHA)

            # wait until info is available
            CLASE_CONTENEDOR_VUELO = "Fxw9-result-item-container"
            vuelos_finales = []
            elementos_vuelo = []   

            if esperar_carga_vuelos(driver, CLASE_CONTENEDOR_VUELO):
                # Obtain container list
                elementos_vuelo = extraer_contenedores(driver, CLASE_CONTENEDOR_VUELO)

                # Iteration and filtering
                print("\n--- INICIANDO EXTRACCIÓN Y FILTRADO DE ANUNCIOS ---")
                for i, contenedor in enumerate(elementos_vuelo):
                    # Date is global date previously obtained
                    datos = extraer_datos_vuelo_completo(contenedor, fecha_de_busqueda)

                    if datos.get('tipo') == 'VUELO':
                        datos.pop('tipo', None)
                        datos.setdefault('origen_iata', origin)
                        datos.setdefault('destino_iata', dest)
                        datos.setdefault('fecha_vuelo', dep.isoformat())
                        vuelos_finales.append(datos)

            else:
                print("No se cargaron vuelos") #(esperar_carga_vuelos returned False)

            print(f"\nExtracción finalizada. Total de vuelos guardados (filtrados): {len(vuelos_finales)}")

            # save results
            if vuelos_finales:
                guardar_a_csv(vuelos_finales)
            else:
                print("No hay vuelos para guardar en este intento.")

        except Exception as e:
            print("ERROR →", origin, dest, dep, e)

        # 7. Rate limit
        time.sleep(random.uniform(0, 2))


driver.quit()

In [ ]:
driver.quit()